# 小さなチャンクで検索して、周りの文脈ごと LLM に渡す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_rag/rag_parent_child.ipynb)

NLP クックブック 18-03 ／ RAG (検索拡張生成)

チャンクを小さくすると検索は当たりやすいが、LLM に渡す文脈が足りなくなる。検索の単位と、読ませる単位を分けたい。

In [ ]:
import io, re, urllib.request, zipfile
from sentence_transformers import SentenceTransformer

def aozora(url):
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    text = z.read(z.namelist()[0]).decode("cp932")
    text = re.split(r"-{50,}", text)[2].split("底本：")[0]
    return re.sub(r"《.+?》|［＃.+?］|｜", "", text).replace("\r\n", "\n").strip()

text = aozora("https://www.aozora.gr.jp/cards/000035/files/1567_ruby_4948.zip")   # 走れメロス
sents = [s.strip() for s in re.split(r"(?<=[。！？」])(?![」。！？])|\n+", text) if s and s.strip()]

# 子: 1 文ずつ (検索の単位) / 親: 前後 3 文を含む窓 (LLM に渡す単位)
def parent(i, window=3):
    return "".join(sents[max(0, i - window):i + window + 1])

model = SentenceTransformer("intfloat/multilingual-e5-small")
emb = model.encode(["passage: " + s for s in sents], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
print(len(sents), "文")

In [ ]:
def search(question, k=3):
    sims = emb @ model.encode("query: " + question, normalize_embeddings=True)
    hits, used = [], set()
    for i in sims.argsort()[::-1]:
        if any(abs(int(i) - j) <= 3 for j in used):     # 既に採用した窓と重なる文は飛ばす
            continue
        used.add(int(i)); hits.append((int(i), float(sims[i])))
        if len(hits) == k:
            break
    return hits

for q in ["メロスの妹は誰と結婚するのか", "王はなぜ人を殺すのか"]:
    print(q)
    for i, score in search(q, k=2):
        print(f"  {score:.3f} 当たった文: {sents[i][:50]}")
        print(f"        渡す文脈: {parent(i)[:120]}...")

- 「small-to-big」「parent document retrieval」と呼ばれる方法。1 文〜数文の小さな単位で索引を作り、ヒットした位置の周辺 (段落や節の全体) を LLM に渡す。
- 親の単位は、前後の窓でも、元の段落・節・ページでもよい。構造のある文書なら、見出し単位の節を親にするのが自然。
- 近い位置の文が複数ヒットすると同じ文脈を重複して渡すことになるので、重なりを除く。